In [0]:
import dlt
from pyspark.sql.functions import *



### Bridge Metadata (static)

In [0]:
@dlt.table(
    name="02_silver.bridge_metadata",
    comment="Static data for five major NYC briges"

)

def bridge_metadata():
    bridges = [
        {
            "bridge_id":1,
            "name": "Brooklyn Bridge",
            "length_m": 2460,
            "main_span_m":324,
            "height": 343,
            "location": "New York",
            "type": "Cable-stayed viaduct",
            "opened_year": 2004
        },
        {
            "bridge_id":2,
            "name": "George Washington Bridge",
            "length_m": 4768,
            "main_span_m": 345,
            "height": 416,
            "location": "New York",
            "type": "Suspension viaduct",
            "opened_year": 1998
        },
        {
            "bridge_id":3,
            "name": "Queens Midtown Tunnel",
            "length_m": 2350,
            "main_span_m": 954,
            "height": 354,
            "location": "New York",
            "type": "Tunnel",
            "opened_year": 2022
        },
        {
            "bridge_id":4,
            "name": "Thames Tunnel",
            "length_m": 7845,
            "main_span_m": 234,
            "height": 965,
            "location": "London",
            "type": "Cable-stayed & Tunnel",
            "opened_year": 1944
        },
        {
            "bridge_id":5,
            "name": "Tower Bridge",
            "length_m": 644,
            "main_span_m": 234,
            "height": 343,
            "location": "London",
            "type": "Cable-stayed & Tunnel",
            "opened_year": 1890
        }

    ]
    return spark.createDataFrame(bridges)


### Bridge Temeprature (Streaming Table)

In [0]:
@dlt.table(
    name="02_silver.bridge_temperature", 
    comment="Temperature enriched with metadata"
    )

@dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
@dlt.expect("valid_temperature", "temperature BETWEEN -20 AND 60")
def silver_bridge_temperature():
    return(
        spark.readStream.table("01_bronze.bridge_temperature")
        .withColumn("event_time", col("event_time").cast("timestamp"))
        .withColumnRenamed("device_id", "bridge_id")
        .join(
            dlt.read("02_silver.bridge_metadata"),
            on="bridge_id", how="left")
        .select(
            col("bridge_id"),
            col("name"),
            col("location"),
            col("event_time"),
            col("temperature")
            )
        )
    


### Bridge Vibration

In [0]:
@dlt.table(
    name = "02_silver.bridge_vibration",
    comment = "Vibration enriched with metadata"
)
@dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
@dlt.expect("valid_vibration_range", "vibration BETWEEN 0 AND 0.1")
def silver_bridge_vibration():
    return(
        spark.readStream.table("01_bronze.bridge_vibration")
        .withColumn("event_time", col("event_time").cast("timestamp"))
        .withColumnRenamed("device_id", "bridge_id")
        .join(dlt.read("02_silver.bridge_metadata"), on="bridge_id", how="left")
        .select(
            col("bridge_id"),
            col("name"),
            col("location"),
            col("event_time"),
            col("vibration")
            )
        )
    

#### Bridge Tilt (Streaming Table)

In [0]:
from pyspark.sql.types import TimestampType
@dlt.table(
    name = "02_silver.bridge_tilt",
    comment = "Birdge tilt angle enriched with metadata"
)
@dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
@dlt.expect("valid_tilt_range", "tilt_angle BETWEEN -0.005 AND 0.005")
def silver_bridge_tilt():
    return(
        spark.readStream.table("01_bronze.bridge_tilt")
        .withColumn("event_time", col("event_time").cast(TimestampType()))
        .withColumnRenamed("device_id", "bridge_id")
        .join(dlt.read("02_silver.bridge_metadata"), on="bridge_id", how="left")
        .select(
            col("bridge_id"),
            col("name"),
            col("location"),
            col("event_time"),
            col("tilt_angle")
            )
        )
    